In [2]:
# =====================================================
# 1. IMPORT LIBRARIES
# =====================================================
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'iframe'

plt.style.use('dark_background')

In [3]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/krupalpatel07/porsche-automobil-holding-se/Porsche Automobil Holding SE.csv


In [4]:
# =====================================================
# 2. LOAD DATA
# =====================================================
file_path = "/kaggle/input/datasets/krupalpatel07/porsche-automobil-holding-se/Porsche Automobil Holding SE.csv"
df = pd.read_csv(file_path)

In [5]:
# =====================================================
# 3. PREPROCESSING
# =====================================================
df.columns = [c.lower() for c in df.columns]
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.set_index('date', inplace=True)

In [6]:
# =====================================================
# 4. CRIMSON HEADER
# =====================================================
from IPython.display import display, HTML

def crimson_header(text):
    display(HTML(f"""
    <div style="
        background: linear-gradient(90deg, #8B0000, #000000);
        padding: 20px; border-radius: 14px; margin-top:20px;">
        <h1 style="color:#ff4d4d; text-align:center;">{text}</h1>
    </div>
    """))

crimson_header("📊 Price Structure")

In [7]:
# =====================================================
# 5. PRICE VISUAL
# =====================================================
fig = px.line(df, y='close', title='Porsche Price')
fig.show()

In [8]:
# =====================================================
# 6. RETURNS & VOLATILITY
# =====================================================
crimson_header("⚡ Volatility Engine")

df['returns'] = df['close'].pct_change()
df['vol'] = df['returns'].rolling(20).std()

fig = px.line(df, y='vol', title='Rolling Volatility')
fig.show()

In [9]:
# =====================================================
# 7. GAMMA EXPOSURE PROXY
# =====================================================
crimson_header("🧠 Gamma Exposure Proxy")

# proxy: second derivative of price
velocity = df['close'].diff()
df['gamma'] = velocity.diff()

fig = px.line(df, y='gamma', title='Gamma Proxy (2nd Derivative)')
fig.show()

In [10]:
# =====================================================
# 8. CONVEXITY DETECTION
# =====================================================
crimson_header("🌀 Convexity Zones")

# convexity: acceleration normalized
df['convexity'] = df['gamma'] / (1 + velocity**2)

fig = px.line(df, y='convexity', title='Convexity Signal')
fig.show()

In [11]:
# =====================================================
# 9. GAMMA REGIMES
# =====================================================
crimson_header("⚡ Gamma Regimes")

regime = np.where(df['gamma'] > 0, 'Positive Gamma', 'Negative Gamma')
df['regime'] = regime

fig = px.scatter(df, x=df.index, y='close', color='regime')
fig.show()


In [12]:
# =====================================================
# 10. OPTIONS-STYLE SIGNAL
# =====================================================
crimson_header("🎯 Options Logic Signal")

signal = (df['gamma'] > 0) & (df['vol'] < df['vol'].rolling(50).mean())
df['signal'] = signal.astype(int)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['close'], name='Price'))
fig.add_trace(go.Scatter(x=df.index[df['signal']==1], y=df['close'][df['signal']==1],
                         mode='markers', name='Signal'))
fig.show()


In [13]:
# =====================================================
# 11. EXTREME GAMMA EVENTS
# =====================================================
crimson_header("🔥 Extreme Gamma Events")

z = (df['gamma'] - df['gamma'].mean())/df['gamma'].std()
df['extreme'] = np.abs(z) > 2

fig = px.scatter(df, x=df.index, y='close', color='extreme')
fig.show()

In [14]:
# =====================================================
# 12. FINAL INSIGHTS
# =====================================================
crimson_header("📌 Options Intelligence")

print("""
1. Gamma proxy captures acceleration in price moves.
2. Positive gamma aligns with stable trends.
3. Negative gamma indicates unstable zones.
4. Convexity reveals nonlinear behavior.
5. Options-style thinking enhances edge.
""")

# =====================================================
# END
# =====================================================



1. Gamma proxy captures acceleration in price moves.
2. Positive gamma aligns with stable trends.
3. Negative gamma indicates unstable zones.
4. Convexity reveals nonlinear behavior.
5. Options-style thinking enhances edge.

